In [ ]:
import cloudscraper
import os
import time
import requests
from tqdm import tqdm
from bs4 import BeautifulSoup
import re
from urllib.parse import quote, unquote, quote_plus
import random
BASE_URL = "https://www.ebookhunter.net"

# List of user agents to rotate
LIST_OF_USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/60.0.3112.113 Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/60.0.3112.90 Safari/537.36',
    'Mozilla/5.0 (Windows NT 5.1; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/60.0.3112.90 Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.2; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/60.0.3112.90 Safari/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/44.0.2403.157 Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.3; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/60.0.3112.113 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/57.0.2987.133 Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/57.0.2987.133 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/55.0.2883.87 Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/55.0.2883.87 Safari/537.36',
    'Mozilla/4.0 (compatible; MSIE 9.0; Windows NT 6.1)',
    'Mozilla/5.0 (Windows NT 6.1; WOW64; Trident/7.0; rv:11.0) like Gecko',
    'Mozilla/5.0 (compatible; MSIE 9.0; Windows NT 6.1; WOW64; Trident/5.0)',
    'Mozilla/5.0 (Windows NT 6.1; Trident/7.0; rv:11.0) like Gecko',
    'Mozilla/5.0 (Windows NT 6.2; WOW64; Trident/7.0; rv:11.0) like Gecko',
    'Mozilla/5.0 (Windows NT 10.0; WOW64; Trident/7.0; rv:11.0) like Gecko',
    'Mozilla/5.0 (compatible; MSIE 9.0; Windows NT 6.0; Trident/5.0)',
    'Mozilla/5.0 (Windows NT 6.3; WOW64; Trident/7.0; rv:11.0) like Gecko',
    'Mozilla/5.0 (compatible; MSIE 9.0; Windows NT 6.1; Trident/5.0)',
    'Mozilla/5.0 (Windows NT 6.1; Win64; x64; Trident/7.0; rv:11.0) like Gecko',
    'Mozilla/5.0 (compatible; MSIE 10.0; Windows NT 6.1; WOW64; Trident/6.0)',
    'Mozilla/5.0 (compatible; MSIE 10.0; Windows NT 6.1; Trident/6.0)',
    'Mozilla/4.0 (compatible; MSIE 8.0; Windows NT 5.1; Trident/4.0; .NET CLR 2.0.50727; .NET CLR 3.0.4506.2152; .NET CLR 3.5.30729)'
]

In [ ]:

def get_random_headers():
    """Return headers with a random user agent"""
    return {
        "User-Agent": random.choice(LIST_OF_USER_AGENTS),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Accept-Encoding": "gzip, deflate",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
    }


def get_max_pages(soup):
    """Extract the maximum number of pages from pagination"""
    nav_links = soup.find("div", class_="nav-links")
    if not nav_links:
        return 1  # Only one page if no pagination found

    page_numbers = []
    # Find all page number links
    for link in nav_links.find_all("a", class_="page-numbers"):
        try:
            # Extract page number from URL
            href = link.get("href", "")
            if "/page/" in href:
                page_num = int(re.search(r"/page/(\d+)/", href).group(1))
                page_numbers.append(page_num)
        except (AttributeError, ValueError):
            continue

    # Also check for the current page
    current_page = nav_links.find("span", class_="page-numbers current")
    if current_page:
        try:
            page_num = int(current_page.get_text(strip=True))
            page_numbers.append(page_num)
        except ValueError:
            pass

    return max(page_numbers) if page_numbers else 1

In [ ]:
# ...existing code...
import time
import random
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def attach_retry_adapter(session, total=5, backoff_factor=1):
    retry = Retry(
        total=total,
        backoff_factor=backoff_factor,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET", "POST", "HEAD", "OPTIONS"])
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)


def fetch_with_retries(scraper, url, attempts=4, timeout=20):
    attach_retry_adapter(scraper, total=attempts, backoff_factor=0.5)
    last_exc = None
    for attempt in range(1, attempts + 1):
        scraper.headers.update(get_random_headers())
        try:
            resp = scraper.get(url, timeout=timeout)
            resp.raise_for_status()
            return resp
        except requests.RequestException as e:
            last_exc = e
            wait = (2 ** (attempt - 1)) + \
                random.uniform(0, 1)  # exponential + jitter
            print(
                f"[!] Attempt {attempt} failed: {e} — retrying in {wait:.1f}s")
            time.sleep(wait)
    raise last_exc


# usage
scraper = cloudscraper.create_scraper()
query = 'Kasie West'
first_page_url = f"{BASE_URL}/?s={quote_plus(query)}"
print(f"[*] Checking first page: {first_page_url}")

try:
    response = fetch_with_retries(
        scraper, first_page_url, attempts=5, timeout=15)
    print(f"[*] Successfully retrieved first page after retries")
    soup = BeautifulSoup(response.text, "html.parser")
    max_pages = get_max_pages(soup)
    print(f"[*] Found {max_pages} pages of results")
    # Process the first page
    books = soup.find_all("article", class_="post-box")
    for book in books:
        title_tag = book.find("h2", class_="title")
        if title_tag:
            title = title_tag.get_text(strip=True)
            print(f"[*] Found book: {title}")
except requests.RequestException as e:
    print(f"[!] Failed to retrieve first page after retries: {e}")
# ...existing code...

In [ ]:
def downld_epub_fast(epub_link, scraper, book_title, download_dir="download_dir", chunk_size=65536, max_workers=4, max_retries=3):
    """
    Fast EPUB downloader with multiple optimizations:
    - Retry on failure (up to 3 times, 5s delay)
    - Detect and convert ebookhunter short links
    - Larger chunk size (64KB default)
    - Parallel chunk downloading for large files
    - Reduced system calls
    - Optimized file I/O
    """
    attempt = 1
    while attempt <= max_retries:
        try:
            # ✅ Step 1: Normalize link if from theebookhunter
            if epub_link.startswith("https://theebookhunter.com/d?="):
                file_id = epub_link.split("d?=")[-1].strip()
                epub_link = f"https://drive.usercontent.google.com/download?id={file_id}&export=download&authuser=0"
                print(f"🔄 Converted ebookhunter link → {epub_link}")

            os.makedirs(download_dir, exist_ok=True)
            
            # First, get file info with HEAD request (faster than GET for metadata)
            scraper = cloudscraper.create_scraper()
            #scraper.headers.update(get_random_headers())
            head_response = scraper.head(epub_link, timeout=30)
            head_response.raise_for_status()
            
            # Extract filename from Content-Disposition
            cd = head_response.headers.get("content-disposition", "")
            match = re.search(r'filename="?([^"]+)"?', cd)
            if match:
                raw_name = match.group(1)
                final_filename = os.path.basename(raw_name.strip('"'))
            else:
                final_filename = book_title.strip() + ".epub"
                print("❌ No valid filename in headers, using book title")
                
            save_path = os.path.join(download_dir, final_filename)
            
            # Skip if already exists
            if os.path.exists(save_path):
                print(f"⏭️  File already exists: {save_path}")
                return save_path
                
            total_size = int(head_response.headers.get("content-length", 0))
            
            # For small files or when parallel download isn't beneficial, use simple download
            if total_size < 10 * 1024 * 1024:  # Less than 10MB
                return _simple_fast_download(epub_link, scraper, save_path, final_filename, total_size, chunk_size)
            
            # (Parallel download logic would continue here...)
            print("⚡ Parallel download not implemented yet")
            return None

        except Exception as e:
            print(f"❌ Download failed (attempt {attempt}/{max_retries}): {e}")
            if attempt < max_retries:
                print("⏳ Retrying in 5 seconds...")
                time.sleep(5)
                attempt += 1
            else:
                print("🚫 All retry attempts failed.")
                return None

    
    

def _simple_fast_download(epub_link, scraper, save_path, filename, total_size, chunk_size):
    """Optimized single-threaded download for smaller files"""
    with scraper.get(epub_link, stream=True, timeout=30) as response:
        response.raise_for_status()
        
        with open(save_path, "wb") as file, tqdm(
            desc=filename,
            total=total_size,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            # Write chunks in larger batches to reduce system calls
            buffer = bytearray()
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    buffer.extend(chunk)
                    # Write buffer when it gets large enough
                    if len(buffer) >= chunk_size * 4:  # Write every ~256KB
                        file.write(buffer)
                        bar.update(len(buffer))
                        buffer.clear()
            
            # Write remaining buffer
            if buffer:
                file.write(buffer)
                bar.update(len(buffer))
    
    print(f"✅ Download complete: {save_path}")
    return save_path



def extract_epub_title(soup):
    """Extract the EPUB title from the h2 inside post-single-content div"""
    content_div = soup.find("div", class_="post-single-content")
    if not content_div:
        return None
        
    # Find the h2 tag inside the content div
    h2_tag = content_div.find("h2")
    if h2_tag:
        title_text = h2_tag.get_text(strip=True)
        # Remove " – Free eBooks Download" from the title
        title_text = re.sub(r'\s*–\s*Free\s*eBooks?\s*Download\s*$', '', title_text, flags=re.IGNORECASE)
        return title_text
    
    return None


def get_book_details(book_url, scraper):
    """Scrape book details and download link(s) from an individual book page."""
    try:
        response = fetch_with_retries(scraper, book_url, attempts=5, timeout=15)
        response.raise_for_status()
    except requests.RequestException as e:
        print(f"[!] Failed to retrieve book page {book_url}: {e}")
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    # Title - try to get from the h2 in post-single-content first
    epub_title = extract_epub_title(soup)
    if not epub_title:
        # Fall back to the regular title
        title_tag = soup.find("h1", class_="title")
        epub_title = title_tag.get_text(strip=True) if title_tag else "Unknown Title"

    # Author
    author = 'Unknown'
    title = 'Unknown'
    if 'by' in epub_title.lower():
        parts = epub_title.split("by")
        title = parts[0].strip()
        author = parts[1].strip() if len(parts) > 1 else 'Unknown'
    else:
        author = 'Unknown'
        


    # Download links (inside post-single-content)
    content = soup.find("div", class_="post-single-content")
    download_links = []
    if content:
        for a in content.find_all("a", href=True):
            href = a["href"].strip()
            if href.startswith("//"):
                href = "https:" + href
            # More comprehensive link detection
            
            download_links.append(href)

    return {
        "Title": title,
        "Author": author,
        "DownloadLinks": download_links,
        "URL": book_url
    }



def search_ebooknet_books(query, max_pages=None, delay=2, auto_download=True, download_dir="downloads"):
    """Search for books and return details (and optionally download EPUBs)."""
    results = []
    seen_links = set()
    scraper = cloudscraper.create_scraper()

    # First, get the first page to determine max pages
    first_page_url = f"{BASE_URL}/?s={quote_plus(query)}"
    print(f"[*] Checking first page: {first_page_url}")
    
    try:
        response = fetch_with_retries(scraper, first_page_url, attempts=5, timeout=15)
        response.raise_for_status()
    except requests.RequestException as e:
        print(f"[!] Failed to retrieve first page: {e}")
        return results

    soup = BeautifulSoup(response.text, "html.parser")
    
    # Determine max pages if not specified
    if max_pages is None:
        max_pages = get_max_pages(soup)
        print(f"[*] Found {max_pages} pages of results")
    else:
        print(f"[*] Limiting to {max_pages} pages")

    # Process the first page
    books = soup.find_all("article", class_="post-box")
    for book in books:
        title_tag = book.find("h2", class_="title")
        link_tag = title_tag.find("a") if title_tag else None

        if link_tag:
            book_link = link_tag.get("href", "").strip()
            if book_link and book_link not in seen_links:
                seen_links.add(book_link)
                print(f"    [+] Fetching book details: {book_link}")

                details = get_book_details(book_link, scraper)
                if details:
                    results.append(details)
                    print(f"    [+] Found book details: {details['Title']} by {details['Author']} links: {len(details['DownloadLinks'])}")

                    # Auto download EPUB (first link only)
                    if details['DownloadLinks']:
                        epub_link = details['DownloadLinks'][0]
                        book_title = details['Title'] + " by " + details['Author']
                        print(f"    [↓] Downloading EPUB: {epub_link}")
                        downld_epub_fast(epub_link, scraper, book_title, download_dir=download_dir)

                # Add variable delay to avoid detection
                time.sleep(10)

    # Process remaining pages if needed
    for page in range(2, max_pages + 1):
        search_url = f"{BASE_URL}/page/{page}/?s={quote_plus(query)}"
        print(f"\n[*] Scraping search page {page}: {search_url}")

        try:
            response = fetch_with_retries(scraper, search_url, attempts=5, timeout=15)
            response.raise_for_status()
        except requests.RequestException as e:
            print(f"[!] Failed to retrieve search page {page}: {e}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        books = soup.find_all("article", class_="post-box")

        if not books:
            print("[!] No more results found.")
            break

        for book in books:
            title_tag = book.find("h2", class_="title")
            link_tag = title_tag.find("a") if title_tag else None

            if link_tag:
                book_link = link_tag.get("href", "").strip()
                if book_link and book_link not in seen_links:
                    seen_links.add(book_link)
                    print(f"    [+] Fetching book details: {book_link}")

                    details = get_book_details(book_link, scraper)
                    if details:
                        results.append(details)

                        print(f"    [+] Found book details: {details['Title']} by {details['Author']} links: {len(details['DownloadLinks'])}")
                        # Auto download EPUB (first link only)
                        if details['DownloadLinks']:
                            epub_link = details['DownloadLinks'][0]
                            book_title = details['Title'] + " by " + details['Author']
                            print(f"    [↓] Downloading EPUB: {epub_link}")
                            downld_epub_fast(epub_link, scraper, book_title, download_dir=download_dir)
                            
                    # Add variable delay to avoid detection
                    time.sleep(10)

        # Randomize delay between pages
        time.sleep(10)

    return results

In [ ]:
search_ebooknet_books("Sienna Mynx", max_pages=1)